In [1]:
pip install openai

In [2]:
pip install python-dotenv

In [3]:
pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentel

In [4]:
import chromadb
import uuid

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

In [6]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("LLM_API_KEY")
)

In [8]:
class MultiLLM:
    def __init__(self):
        self.client = client

        self.models = {
            "responder": "deepseek-ai/deepseek-v4-flash",
            "critic": "deepseek-ai/deepseek-v4-flash",
            "judge": "deepseek-ai/deepseek-v4-flash"
        }

    def _call(self, role, system_prompt, user_prompt):
        response = self.client.chat.completions.create(
            model=self.models[role],
            messages=[
                {
                    "role": "user",
                    "content": f"{system_prompt}\n\n{user_prompt}"
                }
            ],
            temperature=0.3,
            timeout=20
        )

        return response.choices[0].message.content

    def responder(self, system_prompt, user_prompt):
        return self._call("responder", system_prompt, user_prompt)

    def critic(self, system_prompt, user_prompt):
        return self._call("critic", system_prompt, user_prompt)

    def judge(self, system_prompt, user_prompt):
        return self._call("judge", system_prompt, user_prompt)

In [9]:
# Short Term Memory (STM)
stm = []
MAX_HISTORY = 5

def update_stm(role, content):
    stm.append({"role": role, "content": content})

    if len(stm) > MAX_HISTORY:
        stm.pop(0)

In [10]:
# Long Term Memory (LTM)
chroma_client = chromadb.Client()
memory_collection = chroma_client.create_collection(name="memory")

In [11]:
def store_memory(text):
    chunks = text.split("\n")  # split by bullet / lines

    for chunk in chunks:
        chunk = chunk.strip()

        if chunk:  # ignore empty lines
            memory_collection.add(
                documents=[chunk],
                ids=[str(uuid.uuid4())]
            )

In [12]:
def retrieve_memory(query, k=2):
    results = memory_collection.query(
        query_texts=[query],
        n_results=k
    )

    if not results["documents"]:
        return []

    return results["documents"][0]

In [13]:
memory_prompt = """
Extract useful long-term memory from the user input.

Only include:
- stable facts
- preferences
- identity
- goals

Ignore:
- greetings
- temporary info
- one-time questions

Return:
- "NONE" if nothing useful
- otherwise short bullet points

User input:
"""

def extract_memory(user_input):
    raw = llm.responder(memory_prompt, user_input)

    if raw.strip().upper() == "NONE":
        return None

    return raw

In [14]:
def build_context(user_query):
    memories = retrieve_memory(user_query)

    print("\n[MEMORY RETRIEVED]:", memories)
    print("[STM]:", stm)

    memory_block = "\n".join(memories) if memories else "None"
    stm_block = "\n".join([f"{m['role']}: {m['content']}" for m in stm])

    return f"""
Relevant memory:
{memory_block}

Recent conversation:
{stm_block}

Current query:
{user_query}
"""

In [15]:
responder_prompt = '''
You generate, defend, and improve answers.

Inputs:
- user_query
- current_answer
- latest_critique

Rules:
- If no current_answer → generate answer from user_query
- If critique exists → improve answer but also defend strong points
- Do not blindly accept critique
- Be confident but open-minded
- Refine, expand, and justify where needed
- Avoid unnecessary length

- First understand the core issue.
- Give direct, logical, actionable advice.
- Avoid vague or motivational language.
- Break complex problems into simple steps when needed.

Response length (strict):
- If the query is short → respond in 1–3 concise sentences.
- If the query is detailed → give a structured, step-by-step answer.
- Do not over-explain simple queries.
- Do not under-explain complex queries.

- If critique is partially incorrect, explicitly reject incorrect parts
- Do not blindly incorporate all critique

Output format (strict JSON):
{
  "answer": "refined answer",
  "stance": "DEFENDED | MODIFIED"
}
'''

In [16]:
critic_prompt = '''
You are a strict critic.

Inputs:
- user_query
- current_answer

Rules:
- Point out flaws, gaps, weak reasoning, missing details
- Be direct and argumentative
- Do not rewrite full answer
- If answer is mostly correct, acknowledge briefly but still test it
- If no major issue → say it is acceptable but mention minor improvement if any
- Avoid repeating same points

If the answer is correct, DO NOT criticize tone, friendliness, or style.
Only point out factual or logical issues.

Do not suggest unnecessary improvements.

Critique policy:
- Focus ONLY on meaningful issues (logic, correctness, relevance).
- Ignore punctuation, tone, or stylistic preferences.
- Do NOT nitpick minor details like punctuation or wording.
- If the answer is correct and appropriate, say: "No major issues".

Output format (strict JSON):
{
  "critique": "direct critique text",
  "severity": "LOW | MEDIUM | HIGH"
}
'''

In [17]:
judge_prompt = '''
You evaluate whether a critique is worth applying.

Inputs:
- user_query
- current_answer
- latest_critique

Rules:
- Focus only on correctness, relevance, and clarity.
- Ignore tone, style, wording, punctuation.
- Useful critique = identifies real issue (logic, facts, relevance).
- Not useful = nitpicking, overthinking, or unnecessary changes.

Special case:
- If query is a simple greeting (e.g., "hi", "hello"):
  → Any valid greeting response is sufficient
  → use_critique = false

Decision:
- If critique improves answer meaningfully → use_critique = true
- Otherwise → use_critique = false

Output (strict JSON):
{
  "use_critique": true/false,
  "confidence": 0-1,
  "reason": "short reason"
}'''

In [18]:
llm = MultiLLM()

In [19]:
current_answer = ""
latest_critique = ""
judge_feedback = ""

In [20]:
user_prompt = input("Enter your Prompt: ")

Enter your Prompt: Hello


In [21]:
import json
import re

def extract_json(text):
    try:
        return json.loads(text)
    except:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
        raise ValueError("No valid JSON found")

In [22]:
def run_responder():
    global current_answer

    context = build_context(user_prompt)

    responder_input = f"""
    user_query: {context}
    current_answer: {current_answer}
    latest_critique: {latest_critique}
    """

    raw = llm.responder(responder_prompt, responder_input)
    print("\n[RAW RESPONDER OUTPUT]:\n", raw)

    try:
        data = extract_json(raw)
        print("[PARSED]:", data)

        current_answer = data.get("answer", current_answer)

        if latest_critique == "":
            print("[ANSWER]:", current_answer)
        else:
            print("[REFINED]:", current_answer)

    except Exception as e:
        print("[ERROR] Responder failed:", e)

In [23]:
def run_critic():
    global latest_critique

    context = build_context(user_prompt)

    critic_input = f"""
    user_query: {context}
    current_answer: {current_answer}
    """

    raw = llm.critic(critic_prompt, critic_input)
    print("\n[RAW CRITIC OUTPUT]:\n", raw)

    try:
        data = extract_json(raw)
        print("[PARSED]:", data)

        latest_critique = data.get("critique", "")
        print("[CRITIQUE]:", latest_critique)

    except Exception as e:
        print("[ERROR] Critic failed:", e)

In [24]:
def run_improve():
    global current_answer

    context = build_context(user_prompt)

    responder_input = f"""
    user_query: {context}
    current_answer: {current_answer}
    latest_critique: {latest_critique}
    """

    response = llm.responder(responder_prompt, responder_input)

    try:
        data = extract_json(response)
        current_answer = data.get("answer", current_answer)
        print("[IMPROVED]:", current_answer)
    except:
        print("[ERROR] Improve failed")

In [27]:
def run_judge():
    global judge_output

    context = build_context(user_prompt)

    judge_input = f"""
    user_query: {context}
    current_answer: {current_answer}
    """

    raw = llm.judge(judge_prompt, judge_input)
    print("\n[RAW JUDGE OUTPUT]:\n", raw)

    try:
        data = extract_json(raw)
        print("[PARSED]:", data)

        judge_output = data
        print("[JUDGE]:", judge_output)

    except Exception as e:
        print("[ERROR] Judge failed:", e)
        judge_output = {"use_critique": False}  # safe fallback

In [28]:
max_iters = 5
judge_output = {"use_critique": True}

update_stm("user", user_prompt)
context = build_context(user_prompt)

for i in range(max_iters):
    print(f"\n======== ITERATION {i+1} ========")

    if i == 0:
        run_responder()
    else:
        run_improve()

    run_critic()
    run_judge()

    if not judge_output.get("use_critique", True):
        print("\n[JUDGE STOPPED ITERATION]")
        break

    if i == max_iters - 1:
        print("\n[FORCED STOP]: Max iterations reached")
        break

print(f"\nFinal response: {current_answer}")

update_stm("assistant", current_answer)

memory = extract_memory(user_prompt)

if memory:
    store_memory(memory)
    print("\n[MEMORY STORED]:", memory)


[MEMORY RETRIEVED]: []
[STM]: [{'role': 'user', 'content': 'Hello'}, {'role': 'user', 'content': 'Hello'}]

======== ITERATION 1 ========

[MEMORY RETRIEVED]: []
[STM]: [{'role': 'user', 'content': 'Hello'}, {'role': 'user', 'content': 'Hello'}]

[RAW RESPONDER OUTPUT]:
 {
  "answer": "Hello! How can I assist you today?",
  "stance": "DEFENDED"
}
[PARSED]: {'answer': 'Hello! How can I assist you today?', 'stance': 'DEFENDED'}
[REFINED]: Hello! How can I assist you today?

[MEMORY RETRIEVED]: []
[STM]: [{'role': 'user', 'content': 'Hello'}, {'role': 'user', 'content': 'Hello'}]

[RAW CRITIC OUTPUT]:
 {
  "critique": "The user's query is simply 'Hello', and the response is a generic greeting. There is no factual or logical issue here, as the assistant correctly acknowledges the greeting and offers assistance. No meaningful flaws or gaps exist.",
  "severity": "LOW"
}
[PARSED]: {'critique': "The user's query is simply 'Hello', and the response is a generic greeting. There is no factual o

# v2 (experimental)

Changes:
- Added multi-agent loop (responder, critic, judge)
- Implemented iterative refinement logic
- Added short-term memory (STM)
- Added long-term memory (ChromaDB)
- Context building with memory + history
- Basic memory extraction system
- Chunked memory storage
- Improved loop control (judge-based stopping)

Known issues:
- API calls can hang (no timeout yet)
- Performance not optimized
- Needs better error handling

v3:
- Add timeout + retry
- Optimize latency
- Improve reliability